In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("=" * 65)
print("BANK TRANSACTION — EXPLORATORY DATA ANALYSIS")
print("=" * 65)

BANK TRANSACTION — EXPLORATORY DATA ANALYSIS


In [5]:
# ─── STEP 1: Load the data ──────────────────────────────────────
df = pd.read_csv(r'C:\Users\DELL\Downloads\Jupyter\bank_transactions_data_.csv')
df.columns = [c.strip().replace(' ','_').lower() for c in df.columns]
df['transactiondate'] = pd.to_datetime(df['transactiondate'], errors='coerce')
print(f"\n[LOADED] {df.shape[0]:,} rows × {df.shape[1]} columns")


[LOADED] 50,000 rows × 15 columns


In [6]:
# ─── STEP 2: Data type check ────────────────────────────────────
print("\n[TYPES]")
print(df.dtypes.to_string())
print(f"\nMemory: {df.memory_usage(deep=True).sum()/1024:.1f} KB")


[TYPES]
transactionid                  object
accountid                      object
transactionamount             float64
transactiondate        datetime64[ns]
transactiontype                object
location                       object
deviceid                       object
ip_address                     object
merchantid                     object
channel                        object
customerage                     int64
customeroccupation             object
transactionduration             int64
loginattempts                   int64
accountbalance                float64

Memory: 27059.9 KB


In [7]:
# ─── STEP 3: Missing values ─────────────────────────────────────
print("\n[NULL CHECK]")
null_df = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().sum()/len(df)*100).round(2)
})
print(null_df[null_df['null_count']>0] if null_df['null_count'].sum()>0
      else "  ✓ No nulls found — dataset is complete.")


[NULL CHECK]
                 null_count  null_pct
transactiondate       47511     95.02


In [8]:
# ─── STEP 4: Duplicate check ────────────────────────────────────
dupes = df['transactionid'].duplicated().sum()
print(f"\n[DUPLICATES] {dupes} duplicate TransactionIDs")


[DUPLICATES] 0 duplicate TransactionIDs


In [9]:
# ─── STEP 5: Descriptive statistics ─────────────────────────────
print("\n[DESCRIPTIVE STATISTICS]")
num_cols = ['transactionamount','customerage','transactionduration',
            'loginattempts','accountbalance']
print(df[num_cols].describe().round(2).to_string())
print("\nSkewness:")
print(df[num_cols].skew().round(3).to_string())


[DESCRIPTIVE STATISTICS]
       transactionamount  customerage  transactionduration  loginattempts  accountbalance
count           50000.00     50000.00             50000.00       50000.00        50000.00
mean              297.87        44.65               118.97           1.13         5122.74
std               292.82        17.80                70.00           0.61         3904.27
min                 0.24        18.00                10.00           1.00          101.25
25%                82.90        27.00                63.00           1.00         1508.72
50%               209.36        45.00               111.00           1.00         4735.41
75%               409.62        59.00               161.00           1.00         7713.67
max              2060.59        80.00               300.00           5.00        14977.99

Skewness:
transactionamount      1.765
customerage            0.151
transactionduration    0.616
loginattempts          5.124
accountbalance         0.595


In [10]:
# ───Step 6:Feature engineering ────────────────────────────────
print("\n[FEATURE ENGINEERING]")
df['txn_year']    = df['transactiondate'].dt.year
df['txn_month']   = df['transactiondate'].dt.month
df['txn_quarter'] = df['transactiondate'].dt.quarter
df['day_of_week'] = df['transactiondate'].dt.dayofweek
df['is_weekend']  = df['day_of_week'].isin([5,6])
df['amount_band'] = pd.cut(df['transactionamount'],
    bins=[0,50,200,500,1000,np.inf],
    labels=['Micro','Small','Medium','Large','High Value'])
df['age_group'] = pd.cut(df['customerage'], bins=[17,24,39,54,64,80],
    labels=['Gen Z','Millennial','Gen X','Boomer','Senior'])
df['login_risk'] = np.where(df['loginattempts']>=4,'Critical',
                   np.where(df['loginattempts']==3,'High',
                   np.where(df['loginattempts']==2,'Medium','Normal')))
df['is_suspicious'] = ((df['transactiontype']=='Debit') &
                       (df['transactionamount']>1000) &
                       (df['loginattempts']>1))
print("  Created: txn_year, txn_month, txn_quarter, day_of_week, is_weekend,")
print("           amount_band, age_group, login_risk, is_suspicious")


[FEATURE ENGINEERING]
  Created: txn_year, txn_month, txn_quarter, day_of_week, is_weekend,
           amount_band, age_group, login_risk, is_suspicious


In [11]:
# ─── STEP 7: Categorical distributions ─────────────────────────
print("\n[CATEGORICAL DISTRIBUTIONS]")
for col in ['transactiontype','channel','customeroccupation']:
    print(f"\n{col}:")
    vc = df[col].value_counts(normalize=True).mul(100).round(2)
    for k,v in vc.items():
        print(f"  {k:<20}: {v:.1f}%")


[CATEGORICAL DISTRIBUTIONS]

transactiontype:
  Debit               : 77.5%
  Credit              : 22.5%

channel:
  Branch              : 34.6%
  ATM                 : 33.1%
  Online              : 32.3%

customeroccupation:
  Student             : 26.1%
  Doctor              : 25.2%
  Engineer            : 25.0%
  Retired             : 23.7%


In [12]:
# ─── STEP 8: Key summary statistics ─────────────────────────────
print("\n" + "="*65)
print("KEY EDA FINDINGS")
print("="*65)
print(f"  Total transactions:  {len(df):,}")
print(f"  Unique accounts:     {df['accountid'].nunique():,}")
print(f"  Date range:          {df['transactiondate'].min().date()} → {df['transactiondate'].max().date()}")
print(f"  Total volume (USD):  ${df['transactionamount'].sum():,.2f}")
print(f"  Avg transaction:     ${df['transactionamount'].mean():.2f}")
print(f"  Median transaction:  ${df['transactionamount'].median():.2f}")
print(f"  Skewness (amount):   {df['transactionamount'].skew():.3f} (right-skewed)")
print(f"  Suspicious txns:     {df['is_suspicious'].sum():,} ({df['is_suspicious'].mean()*100:.3f}%)")
print(f"  High-risk logins:    {(df['loginattempts']>=3).sum():,} ({(df['loginattempts']>=3).mean()*100:.2f}%)")



KEY EDA FINDINGS
  Total transactions:  50,000
  Unique accounts:     495
  Date range:          2020-01-02 → 2025-12-11
  Total volume (USD):  $14,893,610.69
  Avg transaction:     $297.87
  Median transaction:  $209.36
  Skewness (amount):   1.765 (right-skewed)
  Suspicious txns:     33 (0.066%)
  High-risk logins:    1,924 (3.85%)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv(r'C:\Users\DELL\Downloads\Jupyter\bank_transactions_data_.csv',
                 parse_dates=['transactiondate'])
print("=" * 65)
print("STATISTICAL ANALYSIS — BANK TRANSACTIONS")
print("=" * 65)

In [18]:
# ─── 1. Normality Tests ─────────────────────────────────────────
print("\n1. NORMALITY TESTS (Shapiro-Wilk, 500-row sample)")
for ch in df['channel'].unique():
    sample = df[df['channel']==ch]['transactionamount'].sample(500, random_state=42)
    stat, p = stats.shapiro(sample)
    res = "NOT Normal" if p < 0.05 else "Normal"
    print(f"   {ch:<8}: W={stat:.4f}, p={p:.6f}  → {res}")


1. NORMALITY TESTS (Shapiro-Wilk, 500-row sample)
   ATM     : W=0.8078, p=0.000000  → NOT Normal
   Online  : W=0.8030, p=0.000000  → NOT Normal
   Branch  : W=0.8139, p=0.000000  → NOT Normal


In [19]:
# ─── 2. A/B Test: Online vs Branch ─────────────────────────────
print("\n2. A/B TEST — Online vs Branch Transaction Amount")
print("   H0: Mean amounts are equal")
print("   H1: Means differ")
print("   Alpha: 0.05")

ga = df[df['channel']=='Online']['transactionamount'].dropna()
gb = df[df['channel']=='Branch']['transactionamount'].dropna()

# Welch's t-test (does NOT assume equal variances — more robust)
t, p = stats.ttest_ind(ga, gb, equal_var=False)

# Cohen's d: practical effect size
d = (ga.mean() - gb.mean()) / np.sqrt((ga.std()**2 + gb.std()**2) / 2)

# Mann-Whitney U (non-parametric — no normality assumption)
u, pu = stats.mannwhitneyu(ga, gb, alternative='two-sided')

print(f"\n   Online (A): n={len(ga):,}  mean=${ga.mean():.2f}  std=${ga.std():.2f}")
print(f"   Branch (B): n={len(gb):,}  mean=${gb.mean():.2f}  std=${gb.std():.2f}")
print(f"\n   Welch t-test:  t={t:.4f},  p={p:.6f}")
print(f"   Mann-Whitney:  U={u:.0f},  p={pu:.6f}")
print(f"   Cohen's d:     {d:.4f}  ({'Large' if abs(d)>=0.8 else 'Medium' if abs(d)>=0.5 else 'Small' if abs(d)>=0.2 else 'Negligible'} effect)")
print(f"\n   RESULT: {'REJECT H0 — significant difference' if p<0.05 else 'FAIL TO REJECT H0'}")
print(f"   Interpretation: The $6.88 difference is statistically real but")
print(f"   Cohen's d={d:.3f} means the PRACTICAL impact is negligible.")
print(f"   Recommendation: Do NOT close branches based on this metric alone.")



2. A/B TEST — Online vs Branch Transaction Amount
   H0: Mean amounts are equal
   H1: Means differ
   Alpha: 0.05

   Online (A): n=16,170  mean=$298.86  std=$295.95
   Branch (B): n=17,278  mean=$291.98  std=$284.64

   Welch t-test:  t=2.1646,  p=0.030422
   Mann-Whitney:  U=140562383,  p=0.324338
   Cohen's d:     0.0237  (Negligible effect)

   RESULT: REJECT H0 — significant difference
   Interpretation: The $6.88 difference is statistically real but
   Cohen's d=0.024 means the PRACTICAL impact is negligible.
   Recommendation: Do NOT close branches based on this metric alone.


In [20]:
# ─── 3. One-Way ANOVA: Login Attempts by Occupation ─────────────
print("\n3. ONE-WAY ANOVA — Login Attempts by Occupation")
print("   H0: All occupations have equal mean login attempts")
print("   H1: At least one occupation differs")
groups = [df[df['customeroccupation']==o]['loginattempts'].values
          for o in sorted(df['customeroccupation'].unique())]
F, pF = stats.f_oneway(*groups)
print(f"   F={F:.4f}, p={pF:.6f}")
print(f"   RESULT: {'REJECT H0 — occupation affects login behaviour' if pF<0.05 else 'No significant difference'}")
for occ in sorted(df['customeroccupation'].unique()):
    g = df[df['customeroccupation']==occ]['loginattempts']
    print(f"   {occ:<12}: mean={g.mean():.3f}  std={g.std():.3f}  n={len(g):,}")



3. ONE-WAY ANOVA — Login Attempts by Occupation
   H0: All occupations have equal mean login attempts
   H1: At least one occupation differs
   F=5.6643, p=0.000710
   RESULT: REJECT H0 — occupation affects login behaviour
   Doctor      : mean=1.141  std=0.646  n=12,578
   Engineer    : mean=1.121  std=0.567  n=12,491
   Retired     : mean=1.134  std=0.662  n=11,872
   Student     : mean=1.112  std=0.560  n=13,059


In [21]:
# ─── 4. Pearson Correlation ─────────────────────────────────────
print("\n4. PEARSON CORRELATIONS WITH TRANSACTION AMOUNT")
num_cols = ['customerage','transactionduration','loginattempts','accountbalance']
for col in num_cols:
    r, rp = stats.pearsonr(df['transactionamount'], df[col])
    sig = '* significant' if rp < 0.05 else ''
    print(f"   vs {col:<28}: r={r:+.4f}  p={rp:.4f}  {sig}")


4. PEARSON CORRELATIONS WITH TRANSACTION AMOUNT
   vs customerage                 : r=-0.0209  p=0.0000  * significant
   vs transactionduration         : r=+0.0134  p=0.0026  * significant
   vs loginattempts               : r=-0.0152  p=0.0007  * significant
   vs accountbalance              : r=-0.0214  p=0.0000  * significant


In [22]:
# ─── 5. Value at Risk ────────────────────────────────────────────
print("\n5. VALUE AT RISK (Historical Simulation)")
print("   VaR95 = worst amount we expect to lose on 95% of transactions")
print("   (5th percentile of the distribution)")
for ch in ['Online','ATM','Branch']:
    data = df[df['channel']==ch]['transactionamount']
    var95 = np.percentile(data, 5)
    var99 = np.percentile(data, 1)
    cvar  = data[data <= var95].mean()
    print(f"   {ch:<8}: VaR95=${var95:.2f}  VaR99=${var99:.2f}  CVaR=${cvar:.2f}")


5. VALUE AT RISK (Historical Simulation)
   VaR95 = worst amount we expect to lose on 95% of transactions
   (5th percentile of the distribution)
   Online  : VaR95=$16.56  VaR99=$4.35  CVaR=$8.42
   ATM     : VaR95=$14.27  VaR99=$3.48  CVaR=$7.84
   Branch  : VaR95=$16.97  VaR99=$5.02  CVaR=$9.54


In [ ]:
# ─── 6. Fraud Pattern Analysis ───────────────────────────────────
print("\n6. FRAUD RISK PATTERN ANALYSIS")
sus  = df[df['is_suspicious']==True]
norm = df[df['is_suspicious']==False]
print(f"   Suspicious transactions: {len(sus):,} ({len(sus)/len(df)*100:.3f}%)")
print(f"   Suspicious avg amount:   ${sus['transactionamount'].mean():.2f}")
print(f"   Normal avg amount:       ${norm['transactionamount'].mean():.2f}")
if len(sus) > 2:
    t2, p2 = stats.ttest_ind(sus['transactionamount'],
                              norm['transactionamount'], equal_var=False)
    print(f"   Suspicious vs Normal:    t={t2:.4f}, p={p2:.6f}")




6. FRAUD RISK PATTERN ANALYSIS
   Suspicious transactions: 33 (0.066%)
   Suspicious avg amount:   $1413.24
   Normal avg amount:       $297.14
   Suspicious vs Normal:    t=41.8800, p=0.000000

[DONE] Statistical analysis complete.
